# ECS CRF-path I/O capture

Captures the **actual inputs and outputs of every step** of the production ECS pipeline for a
single uploaded CRF (`384-201-00002_Annotated Unique CRF`), so we can see exactly what each
stage produces — most importantly the **digitized `{form_name, field_name, field_oid}` records**
and the **OID-subset reuse decisions** — and screenshot them for the pipeline doc.

The pipeline (the one path that runs in production: **upload CRF → digitize → match & generate**)
has these stages, each backed by a real module in the project's `python/` tree:

| Stage | Module / function called | What we capture |
|-------|--------------------------|-----------------|
| **1. Digitize (raw)** | `crf_extraction.HistoricalCRF.historical_mapping()` | per-page OID-page records + data-page records |
| **1. Digitize (merge/join/dedup)** | logic copied verbatim from `crf_extraction.extract_crf_new()` | the flat `{form_name, field_name, field_oid}` list |
| **2. Group per form** | mirrors the `api.py` glue (**not synced in this repo**) | `{form_name, field_oids, fields}` per form → `sub_data` |
| **2. Match & reuse** | `CRFFuzzyMatcher.fuzzy_match_fields()`, subclassed to read the library from the `ecs_index` **dataset** | the final review table (Standard/Historical/LLM Generated rows) |
| **2. Reuse deep-dive** | inherited `get_standard_crf` / `get_all_field_oids_forms` / `is_subset` (same subclass) | fuzzy candidates + library records + subset decision for one form |
| **2. Generate (agent)** | mirrors `generate_ecs_llm()` (agent call, **minus** the Snowflake token INSERT) | the LLM-drafted rules for one leftover form |

**Read-only:** Phase-1 progress writes to Snowflake (`ecs_file_upload`) and the agent's token-log
INSERT are stubbed/skipped, so running this notebook does not mutate any tracking tables.

> **Phase-2 library source.** The live matcher reads its edit-check library from the `ecs_opensearch`
> OpenSearch index via `OpensearchUtil`, which calls `client.get_connection(...).get_info()` — that
> reads the connection's **credentials** and needs admin / connection-ACL rights, so non-admins get
> `UnauthorizedException`. This notebook instead reads the **same** library from the Dataiku dataset
> `ecs_index` (governed by dataset permissions you already have) and subclasses `CRFFuzzyMatcher`,
> overriding **only** the OpenSearch-reading methods. All matching logic is inherited unchanged.

> Note on `api.py`: the doc references an `api.py` that does the flat->grouped transform and calls
> the generation agent, but that file lives on the Dataiku API node and is **not** in this repo.
> The grouping and agent-call cells below reconstruct that glue faithfully and are labelled as such.

In [ ]:
# =============================================================================
# Bootstrap: connect to the ECS project the SAME way the production code does
# (secret-resolved host/key -> DSSClient), and load the real pipeline modules.
# =============================================================================
import io, os, re, ast, json, copy, uuid, time, traceback
from collections import defaultdict, OrderedDict
import pandas as pd
from IPython.display import display

import dataiku, dataikuapi
from utils import connection
from utilities.variables import RD_PROJECT_NAME, SECRET_NAME, TOKEN_KEY

# Real pipeline modules (project `python/` tree is on the path inside a Dataiku notebook).
from crf_extraction_module.crf_extraction import HistoricalCRF, GenericFormFieldExtractor
from crf_extraction_module.review_table import CRFFuzzyMatcher

# --- Client -----------------------------------------------------------------
# The client is used by Phase 1 (the digitizer, LLM calls, managed-folder access)
# and for reading project variables. We do NOT use it for the Phase-2 library read:
# the live matcher gets its library via OpensearchUtil -> client.get_connection(...)
# .get_info(), which reads the connection's CREDENTIALS and needs admin/ACL rights
# (that's the UnauthorizedException non-admins hit). Phase 2 below reads the same
# library from a Dataiku DATASET instead, so it never touches connection details.
# Prefer the in-notebook client (current user); fall back to the secret-based one.
try:
    client = dataiku.api_client()
    proj = client.get_project(RD_PROJECT_NAME)
    _ = proj.get_variables()                      # sanity-check access
    print("client: dataiku.api_client() (in-notebook user identity)")
except Exception as _e:
    print("client: falling back to secret-based DSSClient -", repr(_e))
    DATAIKU_HOST, API_SECRET_KEY = connection.get_dataiku_host_and_api_key(
        RD_PROJECT_NAME, SECRET_NAME, TOKEN_KEY)
    client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
    proj = client.get_project(RD_PROJECT_NAME)

variables = proj.get_variables()["local"]
PROJECT_KEY = proj.project_key

# --- Project variables the production pipeline uses --------------------------
SNOWFLAKE_CONN   = variables.get("snowflake_connection_string")
FILE_UPLOAD_FOL  = variables.get("file_upload")            # managed folder holding uploaded CRFs
GEN_AGENT_ID     = variables.get("ecs_generation_agent_id")

# Resolve the two OpenSearch index names (key -> value, with ${projectKey} substituted).
HIST_INDEX = variables.get("ecs_opensearch")
if HIST_INDEX and "${projectKey}" in HIST_INDEX:
    HIST_INDEX = HIST_INDEX.replace("${projectKey}", PROJECT_KEY).lower()
STANDARD_INDEX = f"{PROJECT_KEY}_ecs_index_data".lower()

# Phase 2 reads the edit-check library from this Dataiku DATASET (the same data the
# `ecs_opensearch` index holds) instead of querying OpenSearch, which avoids the
# connection-credential read that requires admin/ACL. `ecs_index` is the OID-bearing
# library the live matcher uses; `ecs_index_data` is Standard-only + embeddings.
LIBRARY_DATASET = "ecs_index"

# --- What to run -------------------------------------------------------------
# The CRF to capture. The file_upload folder holds MANY files per study (protocols,
# amendments, SAPs, ...), so match on ALL whitespace-separated tokens, not just the
# study id -- otherwise you can silently digitize e.g. a "Protocol Amendment" (which
# has no `Form:` / `field name` markers) and get 0 forms. Keep the aCRF name here.
FILE_QUERY = "384-201-00002 Annotated Unique CRF"

# Study attributes that drive the Historical reuse filter.
# The matcher uses `molecule` FIRST; only if molecule is blank does it fall back to
# `indication`, then `ta`. So set these to the values that actually exist on the
# MSP-2020 Historical records in the library (see tool_output.csv) — otherwise the
# indication/ta fallback would match nothing. molecule=MSP-2020 is the key that
# pulls the reusable Historical rules for this run.
MOLECULE   = "MSP-2020"
INDICATION = "Treatment-Resistant Depression"
TA         = "Neurology"

# Form to deep-dive on (fuzzy candidates + library records + subset decision).
DEEPDIVE_FORM = "Visit Date"

# Optional: run the real generation agent for one LLM-Generated form (makes an LLM call).
RUN_AGENT = False

# Optional: persist all captured artifacts to a managed folder (JSON + CSVs).
SAVE_TO_FOLDER = True
OUTPUT_FOLDER  = "3UkrB0N9"           # any writable managed folder id
OUTPUT_SUBPATH = "ecs_io_capture/384-201-00002"

print("project_key    :", PROJECT_KEY)
print("hist index     :", HIST_INDEX)
print("standard index :", STANDARD_INDEX)
print("file_upload fol:", FILE_UPLOAD_FOL)
print("gen agent id   :", GEN_AGENT_ID)

In [ ]:
# =============================================================================
# Resolve the uploaded CRF inside the file_upload managed folder, and build a
# read-only client wrapper so Phase-1 progress UPDATEs are skipped (no DB writes).
# =============================================================================
_upload_folder = proj.get_managed_folder(FILE_UPLOAD_FOL)
_items = _upload_folder.list_contents().get("items", [])
_paths = [it.get("path") for it in _items]
print(f"{len(_paths)} files in file_upload folder")

# Require EVERY token of FILE_QUERY to appear (case-insensitive), so the study id
# alone can't pull in a protocol/amendment/SAP that shares it.
_tokens = FILE_QUERY.lower().split()
matches = [p for p in _paths if all(tok in (p or "").lower() for tok in _tokens)]
if not matches:
    raise ValueError(f"No file in file_upload folder matches all tokens {_tokens}. "
                     f"First few paths: {_paths[:10]}")
if len(matches) > 1:
    print(f"WARNING: {len(matches)} files match {FILE_QUERY!r} -- picking the first. "
          f"Tighten FILE_QUERY if this isn't the annotated CRF:")
    for m in matches:
        print("   -", m)
FILE_PATH = matches[0]
print("\nresolved FILE_PATH:", FILE_PATH)

# A dummy file_id is fine: it is only used by the progress UPDATE, which we skip.
CAPTURE_FILE_ID = "io-capture-run"


class _NoWriteClient:
    """Wraps the DSSClient but turns sql_query into a no-op, so the digitizer's
    progress writes to ecs_file_upload don't mutate anything during capture.
    Every other attribute delegates to the real client."""
    def __init__(self, real):
        self._real = real
    def sql_query(self, *args, **kwargs):
        print("[capture] skipped a progress sql_query (read-only run)")
        return None
    def __getattr__(self, name):
        return getattr(self._real, name)

In [ ]:
# =============================================================================
# STAGE 1 (raw): run the real digitizer internals on the CRF PDF.
# historical_mapping() classifies each page and returns:
#   response      -> data-page records     {source_data.assessments, source_data.fields}
#   oid_response  -> OID/definition-page records {source_data.assessments, source_data.fields_oid}
# =============================================================================
hcrf = HistoricalCRF(client, proj)
hcrf.client = _NoWriteClient(client)         # read-only: skip progress UPDATEs

t0 = time.time()
response, oid_response = hcrf.historical_mapping(FILE_PATH, CAPTURE_FILE_ID)
print(f"digitized in {time.time()-t0:.1f}s")
print(f"data-page records : {len(response)}")
print(f"OID-page records  : {len(oid_response)}")

print("\n--- sample DATA-PAGE record (fields, no OIDs yet) ---")
print(json.dumps(response[0], indent=2, default=str) if response else "(none)")

print("\n--- sample OID-PAGE record (field_number -> field_oid) ---")
print(json.dumps(oid_response[0], indent=2, default=str) if oid_response else "(none)")

In [ ]:
# =============================================================================
# STAGE 1 (merge/join/dedup): reproduce extract_crf_new()'s post-processing
# VERBATIM (the code below is copied from crf_extraction.extract_crf_new so the
# output is identical to production, without its own progress writes):
#   1. merge fields / OIDs per file+assessment
#   2. join fields to OIDs by field_number  -> each field gets its field_oid
#   3. flatten and dedupe on (form_name, field_name)
# Result: the flat {form_name, field_name, field_oid} list Phase 2 consumes.
# =============================================================================
def merge_sections_per_file(results, field_name):
    merged = defaultdict(dict)
    for item in results:
        file_path = item["path"]
        assessment = item["source_data"]["assessments"]
        if assessment not in merged[file_path]:
            merged[file_path][assessment] = copy.deepcopy(item)
        else:
            merged[file_path][assessment]["source_data"][field_name] += item["source_data"][field_name]
    final = []
    for file_assessments in merged.values():
        final.extend(file_assessments.values())
    return final

ans = merge_sections_per_file(response, "fields")
ans_oid = merge_sections_per_file(oid_response, "fields_oid")

merged_list = []
for d1 in ans:
    matched = False
    for d2 in ans_oid:
        if (d1["template_name"] == d2["template_name"]
                and d1["source_data"]["assessments"] == d2["source_data"]["assessments"]):
            fields = d1["source_data"]["fields"]
            fields_oid = d2["source_data"]["fields_oid"]
            for i, field in enumerate(fields):
                found_oids_name = None
                for each_oids in fields_oid:
                    if int(each_oids["field_number"]) == int(field["field_number"]):
                        found_oids_name = each_oids["field_name"]
                field["field_oid"] = found_oids_name
            merged_list.append(d1)
            matched = True
            break
    if not matched:
        merged_list.append(d1)

final_list = []
for resp in merged_list:
    form_name = resp['source_data']['assessments']
    m = re.search(r'Form[:\s]*(.*)', form_name)
    if m:
        form_name = m.group(1)
    for field in resp['source_data']['fields']:
        field_name = field['field_name']
        field_oid = field['field_oid']
        final_list.append({
            "form_name": form_name,
            "field_name": re.split(r'\t|\s{2,}', field_name.strip())[0],
            "field_oid": field_oid,
        })

seen, digitized = set(), []
for item in final_list:
    key = (item["form_name"].strip().lower(), item["field_name"].strip().lower())
    if key not in seen:
        seen.add(key)
        digitized.append(item)

print(f"flat digitized records (deduped): {len(digitized)}")
digitized_df = pd.DataFrame(digitized)
display(digitized_df.head(20))

# --- The record shape the doc describes, for one form -----------------------
def show_form_records(form_query, n=25):
    rows = [r for r in digitized if form_query.lower() in r["form_name"].lower()]
    print(f"\n=== digitized records for form matching {form_query!r}: {len(rows)} fields ===")
    print(json.dumps(rows[:n], indent=2, default=str))
    return rows

_ = show_form_records(DEEPDIVE_FORM)
_ = show_form_records("Adverse Events")

In [ ]:
# =============================================================================
# STAGE 2 (group): turn the flat digitized list into the per-form bundles the
# matcher expects. This mirrors the api.py glue (NOT synced in this repo):
#   sub_data = [ {form_name, field_oids:[...], fields:[{field_name}, ...]}, ... ]
# Note: fuzzy_match_fields SKIPS any form with an empty field_oids list.
# =============================================================================
groups = OrderedDict()
for item in digitized:
    fn = item["form_name"]
    g = groups.setdefault(fn, {"form_name": fn, "field_oids": [], "fields": []})
    g["fields"].append({"field_name": item["field_name"]})
    if item.get("field_oid"):
        g["field_oids"].append(item["field_oid"])
for g in groups.values():
    g["field_oids"] = list(dict.fromkeys(g["field_oids"]))   # dedupe, preserve order

sub_data = list(groups.values())
skipped = [g["form_name"] for g in sub_data if not g["field_oids"]]

print(f"forms grouped     : {len(sub_data)}")
print(f"forms with 0 OIDs (matcher will skip): {len(skipped)}")
display(pd.DataFrame([
    {"form_name": g["form_name"], "n_fields": len(g["fields"]), "n_field_oids": len(g["field_oids"])}
    for g in sub_data
]))

print(f"\n--- sub_data bundle for {DEEPDIVE_FORM!r} (matcher input) ---")
_dd = next((g for g in sub_data if DEEPDIVE_FORM.lower() in g["form_name"].lower()), None)
print(json.dumps(_dd, indent=2, default=str))

In [ ]:
# =============================================================================
# STAGE 2 (match & reuse): the live matcher reads its edit-check library from the
# OpenSearch index (`ecs_opensearch`) via OpensearchUtil -> client.get_connection()
# .get_info(), which reads the connection CREDENTIALS and needs admin/ACL rights ->
# UnauthorizedException for non-admins. The SAME library is exposed as the Dataiku
# dataset `ecs_index` (governed by dataset permissions, which you have), so we read
# it from there and subclass CRFFuzzyMatcher, overriding ONLY the OpenSearch-reading
# methods. get_standard_crf / is_subset / fuzzy_match_fields are inherited verbatim,
# so the fuzzy-name match, OID-subset reuse and LLM-Generated fallback are identical
# to production.
# =============================================================================
import ast

lib_df = dataiku.Dataset(LIBRARY_DATASET).get_dataframe()
print(f"library rows loaded from dataset {LIBRARY_DATASET!r}: {len(lib_df)}")
print("columns:", list(lib_df.columns))

# Guard: a mis-named/missing `source` or `field_oids` column (or the study-tag columns)
# would make every filter return empty and silently collapse ALL reuse to "LLM Generated"
# with no error. Fail loudly instead so the schema mismatch is obvious.
_required = {"form_name", "source", "molecule", "indication", "ta", "field_oids"}
_missing = _required - set(lib_df.columns)
assert not _missing, (
    f"dataset {LIBRARY_DATASET!r} is missing columns needed for matching: {sorted(_missing)}. "
    f"Available: {list(lib_df.columns)}")
print("source breakdown:", lib_df["source"].value_counts(dropna=False).to_dict())


class DatasetFuzzyMatcher(CRFFuzzyMatcher):
    """CRFFuzzyMatcher whose library comes from a DataFrame instead of OpenSearch.
    Overrides only the index-reading methods; all matching logic is inherited."""

    def __init__(self, df, proj, llm_fallback={"form_field_value": "Not found"},
                 score_threshold=0.40):
        self.df = df.reset_index(drop=True)
        self.index_name = f"{LIBRARY_DATASET} (dataset)"
        self.llm_fallback = llm_fallback
        self.score_threshold = score_threshold
        self._embedding_cache = {}

    @staticmethod
    def _oids_to_str(fo):
        # The inherited fuzzy_match_fields does an UNGUARDED ast.literal_eval(field_oids),
        # so this must ALWAYS return a string that literal-evals to a python list.
        # OpenSearch stored a clean "['A','B']" repr; a dataset cell can instead be an
        # empty string, a bare token, a real list, a numpy array, or NaN.
        if isinstance(fo, str):
            s = fo.strip()
            if not s:
                return "[]"
            try:
                val = ast.literal_eval(s)
            except (ValueError, SyntaxError):
                return "[]"
            return s if isinstance(val, list) else "[]"
        if fo is None or (isinstance(fo, float) and pd.isna(fo)):
            return "[]"
        try:
            return str([str(x) for x in fo])
        except TypeError:
            return "[]"

    def _filter_terms(self, key_name):
        # translate the OpenSearch term-filter list into a pandas mask
        mask = pd.Series(True, index=self.df.index)
        for t in key_name:
            (col_kw, val), = t["term"].items()
            col = col_kw.replace(".keyword", "")
            if col not in self.df.columns:
                return self.df.iloc[0:0]
            mask &= self.df[col].astype(str) == str(val)
        return self.df[mask]

    def get_all_standard_forms(self):
        # mirrors the match_all size=5000 query over the whole index (all sources)
        return self.df["form_name"].dropna().astype(str).head(5000).tolist()

    def get_all_historical_forms(self):
        m = self.df["source"].astype(str) == "Historic"
        return self.df.loc[m, "form_name"].dropna().astype(str).head(1000).tolist()

    def get_all_field_oids_forms(self, form_name, key_name):
        sub = self._filter_terms(key_name)
        if sub.empty:
            return []
        cols = ["validation_id", "ecs_id", "form_id", "form_name", "form_field_value",
                "validation_logic", "reasoning", "indication", "molecule", "ta",
                "field_oids", "action", "source", "action_details", "path"]
        out = []
        for _, item in sub.iterrows():
            rec = {c: item.get(c) for c in cols}
            rec["field_oids"] = self._oids_to_str(item.get("field_oids"))
            out.append(rec)
        return out


matcher = DatasetFuzzyMatcher(lib_df, proj)
print("matcher index_name:", matcher.index_name)

final_answer = matcher.fuzzy_match_fields(sub_data, INDICATION, MOLECULE, TA)
review_df = pd.DataFrame(final_answer)

print(f"\nreview-table rows: {len(review_df)}")
if "source" in review_df.columns:
    print("\nby source:")
    print(review_df["source"].value_counts(dropna=False).to_string())
display(review_df.head(30))

In [ ]:
# =============================================================================
# STAGE 2 (reuse deep-dive): for one form, expose the intermediate reuse steps
# that fuzzy_match_fields does internally, using the SAME matcher methods:
#   1. fuzzy-match the form name to library form names   (get_standard_crf)
#   2. pull candidate Historical + Standard records       (get_all_field_oids_forms)
#   3. is_subset(candidate.field_oids, pooled_study_oids) -> reuse or not
# The subset check uses the study's OIDs POOLED ACROSS ALL FORMS (fecthed_all_field_oids).
# =============================================================================
pooled_oids = sorted({oid for g in sub_data for oid in g["field_oids"]})
print(f"pooled study OIDs (all forms): {len(pooled_oids)}")

all_standard_forms = matcher.get_all_standard_forms()
print(f"library form names loaded: {len(all_standard_forms)}")

dd = next((g for g in sub_data if DEEPDIVE_FORM.lower() in g["form_name"].lower()), None)
assert dd is not None, f"{DEEPDIVE_FORM!r} not found in sub_data"
form_name = dd["form_name"]

candidates = matcher.get_standard_crf(form_name, all_standard_forms) or []
cand_names = list({c[0] for c in candidates})
print(f"\nfuzzy candidates for {form_name!r} (partial_ratio >= 70): {len(cand_names)}")
display(pd.DataFrame(candidates, columns=["library_form", "score", "index"]).head(20))

deepdive = {"form_name": form_name, "pooled_oid_count": len(pooled_oids), "candidates": []}
for cand in cand_names:
    if MOLECULE:
        hist_terms = [{"term": {"molecule.keyword": MOLECULE}},
                      {"term": {"source.keyword": "Historical"}},
                      {"term": {"form_name.keyword": cand}}]
    elif INDICATION:
        hist_terms = [{"term": {"indication.keyword": INDICATION}},
                      {"term": {"source.keyword": "Historical"}},
                      {"term": {"form_name.keyword": cand}}]
    else:
        hist_terms = [{"term": {"ta.keyword": TA}},
                      {"term": {"source.keyword": "Historical"}},
                      {"term": {"form_name.keyword": cand}}]
    std_terms = [{"term": {"source.keyword": "Standard"}},
                 {"term": {"form_name.keyword": cand}}]

    hist_recs = matcher.get_all_field_oids_forms(cand, hist_terms)
    std_recs  = matcher.get_all_field_oids_forms(cand, std_terms)

    def _decide(recs):
        out = []
        for r in recs:
            # NOTE: this try/except is more forgiving than the matcher's inherited
            # fuzzy_match_fields, which calls ast.literal_eval WITHOUT a guard. Because
            # DatasetFuzzyMatcher._oids_to_str now normalizes every field_oids value to a
            # valid list-literal, both paths agree; this guard is just belt-and-braces.
            try:
                oids = ast.literal_eval(r["field_oids"]) if r.get("field_oids") else []
            except Exception:
                oids = []
            out.append({
                "validation_id": r.get("validation_id"),
                "form_field_value": r.get("form_field_value"),
                "source": r.get("source"),
                "molecule": r.get("molecule"),
                "field_oids": oids,
                "is_subset_of_study": bool(oids) and matcher.is_subset(oids, pooled_oids),
            })
        return out

    deepdive["candidates"].append({
        "library_form": cand,
        "historical": _decide(hist_recs),
        "standard": _decide(std_recs),
    })

# Flatten to a table showing which library rules get reused for this form.
rows = []
for c in deepdive["candidates"]:
    for bucket in ("historical", "standard"):
        for r in c[bucket]:
            rows.append({"library_form": c["library_form"], **r})
decision_df = pd.DataFrame(rows)
print(f"\ncandidate library rules for {form_name!r}: {len(decision_df)} "
      f"(reused = is_subset_of_study True)")
display(decision_df)

In [ ]:
# =============================================================================
# STAGE 2 (generate): for ONE form whose fields came back as "LLM Generated"
# placeholders, call the real generation agent and capture the drafted rules.
# This mirrors generate_ecs_llm() EXACTLY except it omits the Snowflake token
# INSERT (kept read-only). Set RUN_AGENT = True in the bootstrap cell to run it.
# =============================================================================
agent_capture = None
if not RUN_AGENT:
    print("RUN_AGENT is False - skipping the live agent call. "
          "Set RUN_AGENT = True in the bootstrap cell to capture generated rules.")
else:
    llm_forms = (review_df.loc[review_df["source"] == "LLM Generated", "form_name"]
                 .dropna().unique().tolist())
    if not llm_forms:
        print("No 'LLM Generated' forms in the review table for this run.")
    else:
        gen_form = llm_forms[0]
        # The placeholder rows carry form_field_value + ecs_id, which is exactly
        # what MyLLM.process()/parse_llm_batch_output expect in `field_name`.
        fields = (review_df.loc[review_df["form_name"] == gen_form,
                                ["form_field_value", "ecs_id"]]
                  .to_dict("records"))
        print(f"Generating for {gen_form!r} with {len(fields)} placeholder fields...")

        payload = {"form_name": gen_form, "field_name": fields}
        completion = proj.get_llm(f"agent:{GEN_AGENT_ID}").new_completion()
        completion.with_context({"payload": payload})
        result = completion.execute()
        parsed = json.loads(result.text)

        agent_capture = {"form_name": gen_form, "payload": payload, "response": parsed}
        print(f"input_token={parsed.get('input_token')} output_token={parsed.get('output_token')}")
        display(pd.DataFrame(parsed.get("response", [])))

In [ ]:
# =============================================================================
# Persist all captured artifacts to a managed folder (JSON + CSV) so they can be
# downloaded / screenshotted for the doc. Set SAVE_TO_FOLDER = False to skip.
# =============================================================================
artifacts = {
    "file_path": FILE_PATH,
    "study": {"molecule": MOLECULE, "indication": INDICATION, "ta": TA},
    "counts": {
        "data_page_records": len(response),
        "oid_page_records": len(oid_response),
        "digitized_fields": len(digitized),
        "forms": len(sub_data),
        "review_rows": len(review_df),
    },
    "digitized": digitized,
    "sub_data": sub_data,
    "deepdive": deepdive,
    "agent_capture": agent_capture,
}

if SAVE_TO_FOLDER and OUTPUT_FOLDER:
    out_fol = proj.get_managed_folder(OUTPUT_FOLDER)

    def _put(path, text):
        out_fol.put_file(f"{OUTPUT_SUBPATH}/{path}", io.BytesIO(text.encode("utf-8")))

    _put("artifacts.json", json.dumps(artifacts, indent=2, default=str))
    _put("digitized.csv", digitized_df.to_csv(index=False))
    _put("review_table.csv", review_df.to_csv(index=False))
    _put(f"deepdive_{re.sub(r'[^A-Za-z0-9]+', '_', DEEPDIVE_FORM)}.csv",
         decision_df.to_csv(index=False))
    print(f"Saved artifacts to folder {OUTPUT_FOLDER} / {OUTPUT_SUBPATH}")
else:
    print("Not saving (SAVE_TO_FOLDER is False or OUTPUT_FOLDER is unset). "
          "Artifacts are available inline via the `artifacts` dict.")

# --- One-line summary --------------------------------------------------------
print("\n=== capture summary ===")
for k, v in artifacts["counts"].items():
    print(f"{k:20s}: {v}")